In [77]:
import pandas as pd

pd.set_option("display.max_columns", 70)

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [78]:
# Convertendo a variavel para numerico, valores que não puderem ser convertidos viram NA
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Clientes com tenure = 0 ainda não possuem cobranças registradas
df.loc[df["tenure"] == 0, "TotalCharges"] = 0

assert df["TotalCharges"].isna().sum() == 0

In [79]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].head()

,tenure,MonthlyCharges,TotalCharges
0,1,29.85,29.85
1,34,56.95,1889.50
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65


## Adicionando as features novas

In [80]:
df["n_protecao"] = (
    (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["OnlineBackup"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
)

In [81]:
df["n_streaming"] = (
    (df["StreamingTV"] == "Yes").astype(int)
    + (df["StreamingMovies"] == "Yes").astype(int)
)

In [82]:
df["n_servicos"] = (
    (df["PhoneService"] == "Yes").astype(int)
    + (df["InternetService"] != "No").astype(int)
    + (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["OnlineBackup"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
    + (df["StreamingTV"] == "Yes").astype(int)
    + (df["StreamingMovies"] == "Yes").astype(int)
)

In [83]:
internet_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

phone_cols = [
    "MultipleLines"
]

df[internet_cols] = df[internet_cols].replace(
    "No internet service",
    "No"
)

df[phone_cols] = df[phone_cols].replace(
    "No phone service",
    "No"
)

In [84]:
# Separando o X e y
X = df.drop(['Churn','customerID'],axis=1)
y = df['Churn']

In [85]:
# Dividindo a base de treino e teste
# O stratify mantém a proporção das classes de Churn nas bases de treino e teste. 

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [86]:
# conferindo a proporção das classes nas bases

print("Treino:", y_train.value_counts(normalize=True))
print("\nTeste:", y_test.value_counts(normalize=True))

Treino: Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64

Teste: Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [87]:
# Separando variaveis numericas e categoricas

num_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

cat_cols = [
    col for col in X_train.columns
    if col not in num_cols
]

In [88]:
print(f"Numéricas: {len(num_cols)}")
print(f"Categóricas: {len(cat_cols)}")

print("\nNuméricas:")
print(num_cols)

print("\nCategóricas:")
print(cat_cols)

Numéricas: 3
Categóricas: 19

Numéricas:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categóricas:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'n_protecao', 'n_streaming', 'n_servicos']


## Enconding nas variáveis categóricas

In [89]:
from sklearn.preprocessing import OneHotEncoder

# Ajustando o encoder apenas nos dados de treino para evitar vazamento de informações .

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_ohe = encoder.fit_transform(X_train[cat_cols])
X_test_ohe = encoder.transform(X_test[cat_cols])

## Scaling

In [90]:
from sklearn.preprocessing import StandardScaler

# O scaler é ajustado apenas na base de treino.
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

## Combinando a base de treino processada

In [91]:
import numpy as np

X_train_processed = np.hstack([
    X_train_num,
    X_train_ohe
])

X_test_processed = np.hstack([
    X_test_num,
    X_test_ohe
])

In [92]:
print("X_train original:", X_train.shape)
print("X_test original:", X_test.shape)

print("X_train processado:", X_train_processed.shape)
print("X_test processado:", X_test_processed.shape)

X_train original: (5634, 22)
X_test original: (1409, 22)
X_train processado: (5634, 55)
X_test processado: (1409, 55)


In [93]:
# Converte o target para o formato binário utilizado pelos modelos.
y_train = y_train.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})


In [94]:
y_test.value_counts()

Churn
0    1035
1     374
Name: count, dtype: int64

In [95]:
feature_names = (
    num_cols
    + list(encoder.get_feature_names_out(cat_cols))
)

len(feature_names)

55

In [96]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

In [97]:
X_train_processed_df.head()

,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,SeniorCitizen_0,SeniorCitizen_1,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_Yes,TechSupport_No,TechSupport_Yes,StreamingTV_No,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,n_protecao_0,n_protecao_1,n_protecao_2,n_protecao_3,n_protecao_4,n_streaming_0,n_streaming_1,n_streaming_2,n_servicos_1,n_servicos_2,n_servicos_3,n_servicos_4,n_servicos_5,n_servicos_6,n_servicos_7,n_servicos_8
0,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,-0.711743,0.337478,-0.503635,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.793155,-0.809013,-0.749883,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.263980,0.284384,-0.172722,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.281624,-0.676279,-0.989374,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [98]:
X_train_processed_df.shape, X_test_processed_df.shape

((5634, 55), (1409, 55))

In [99]:
y_test.value_counts()

Churn
0    1035
1     374
Name: count, dtype: int64

In [100]:
X_train_processed_df.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)